### Simple RAG to answer a question

#### Flow:
1. Add PDF
2. Chunk it & embedd it
3. Store it in a vector database
4. Ask a question & get the answer through LLM

#### Library Imports

In [4]:
from dotenv import load_dotenv
from pathlib import Path
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

#### Load the environment variables

In [5]:
load_dotenv(Path.cwd() / ".env")

True

#### Load the PDF

In [6]:
file_path = "Test_doc.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

#### Chunk the PDF

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_documents(docs)

In [9]:
chunks[0].page_content

'Sai Krishna Reddy Poluri \nData Engineer'

In [10]:
len(chunks)

91

#### Embed the chunks & store in a vector database

In [18]:
persist_directory = str(Path.cwd() / "chroma_db")
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

In [22]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=persist_directory,
    collection_name="rag_test_doc",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

#### Ask a question & get the answer through LLM

In [26]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
question = "Who is Saikrishna?"
docs = retriever.invoke(question)
context = "\n\n".join(d.page_content for d in docs)

from langchain_core.messages import SystemMessage, HumanMessage

llm_response = llm.invoke(
    [
        SystemMessage(
            content="Answer using only the context. If unknown, say you don't know.\n\nContext:\n"
            + context
        ),
        HumanMessage(content=question),
    ]
)
print(llm_response.content)

Sai Krishna Reddy Poluri is a Data Engineer based in Bentonville, AR, with a Bachelor of Technology from SASTRA University, India. He has over 4 years of experience in building scalable distributed architectures and is a Databricks Certified Data Engineer Associate. He has worked as a Data Analyst at Cognizant and as a Business Intelligence Engineer at Manvi Enterprises in India.


In [25]:
# print(context)